# LightGBM + TabTransformer 블렌딩

빠른 검증(1-fold, 3-epoch)에서 LightGBM-TabTransformer OOF 예측의 상관계수가
0.953으로, 다른 트리 계열 모델들(0.97+)보다는 낮게 나왔습니다. 블렌딩 시
LightGBM 80% + TabTransformer 20% 정도에서 단독보다 소폭 개선(+0.0003 수준)을
확인했습니다.

이 노트북은 `1차_tabtransformer(->).ipynb`에서 학습한 TabTransformer(5-Fold,
15 epoch)의 OOF 예측과, main.py의 LightGBM 5-Fold OOF 예측을 블렌딩하여
실제로 의미 있는 개선이 있는지 확인합니다.

**사전 준비**: 이 노트북을 실행하기 전에 `1차_tabtransformer(->).ipynb`을 먼저
끝까지 실행해서 `tt_checkpoints/` 폴더에 5개 fold 체크포인트가 모두 있어야
합니다.


## 1. 라이브러리 및 설정

In [1]:
import os
import numpy as np
import pandas as pd
import torch
torch.set_num_threads(1)
import torch.nn as nn
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings("ignore")
torch.manual_seed(1004)
device = torch.device("cpu")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
CHECKPOINT_DIR = "../tt_checkpoints"

## 2. 데이터 전처리 (main.py / TabTransformer 노트북과 동일)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols and c != TARGET_COL]

cat_cardinalities = []
df_le = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col].astype(str))
    cat_cardinalities.append(df_le[col].nunique())

y = df_le[TARGET_COL].values.astype(np.float32)
X_cat = df_le[cat_cols].values.astype(np.int64)
X_num = df_le[num_cols].values.astype(np.float32)
X_lgbm = df_le.drop(columns=[TARGET_COL])

print(f"전처리 완료: {X_lgbm.shape}")

전처리 완료: (256351, 67)


## 3. TabTransformer 모델 정의 (체크포인트 로드용, 학습 없음)

In [ ]:
class TabTransformer(nn.Module):
    def __init__(self, cat_cardinalities, num_numeric, dim=32, depth=3, heads=4, dropout=0.1):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card + 1, dim) for card in cat_cardinalities])
        self.n_cat = len(cat_cardinalities)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.num_norm = nn.LayerNorm(num_numeric)
        mlp_input_dim = self.n_cat * dim + num_numeric
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1),
        )

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        embs = torch.stack(embs, dim=1)
        embs = self.transformer(embs)
        embs_flat = embs.reshape(embs.size(0), -1)
        x_num_norm = self.num_norm(x_num)
        combined = torch.cat([embs_flat, x_num_norm], dim=1)
        return self.mlp(combined).squeeze(-1)

: 

In [ ]:
import time

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1004)
tr_idx, val_idx = next(skf.split(X_lgbm, y))

t0 = time.time()
m_lgbm = LGBMClassifier(random_state=1004, verbose=-1)
m_lgbm.fit(X_lgbm.iloc[tr_idx], y[tr_idx])
print(f"LightGBM fit: {time.time()-t0:.1f}초")

t1 = time.time()
ckpt = torch.load(f"{CHECKPOINT_DIR}/fold1.pt", map_location=device)
scaler = StandardScaler()
scaler.fit(X_num[tr_idx])
model = TabTransformer(cat_cardinalities, num_numeric=X_num.shape[1]).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(val_idx), 4096):
        batch_idx = val_idx[i:i+4096]
        xb = torch.from_numpy(X_cat[batch_idx]).long().to(device)
        xn = torch.from_numpy(scaler.transform(X_num[batch_idx]).astype(np.float32)).to(device)
        preds.append(torch.sigmoid(model(xb, xn)).cpu().numpy())
print(f"TabTransformer 추론: {time.time()-t1:.1f}초")

## 4. LightGBM + TabTransformer 양쪽 OOF 예측 생성

main.py와 동일한 5-Fold 분할(random_state=1004)을 사용하므로, 저장된
TabTransformer 체크포인트와 fold가 정확히 일치합니다.


In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1004)
oof_lgbm = np.zeros(len(y))
oof_tt = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgbm, y), start=1):
    print(f"=== Fold {fold}/{N_SPLITS} ===")

    # LightGBM
    m_lgbm = LGBMClassifier(random_state=1004, verbose=-1)
    m_lgbm.fit(X_lgbm.iloc[tr_idx], y[tr_idx])
    oof_lgbm[val_idx] = m_lgbm.predict_proba(X_lgbm.iloc[val_idx])[:, 1]
    print(f"  LightGBM AUC: {roc_auc_score(y[val_idx], oof_lgbm[val_idx]):.5f}")

    # TabTransformer (체크포인트에서 로드)
    checkpoint_path = f"{CHECKPOINT_DIR}/fold{fold}.pt"
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"{checkpoint_path}가 없습니다. 1차_tabtransformer(->).ipynb를 먼저 끝까지 실행하세요."
        )
    ckpt = torch.load(checkpoint_path, map_location=device)

    scaler = StandardScaler()
    scaler.fit(X_num[tr_idx])
    X_num_val_s = scaler.transform(X_num[val_idx]).astype(np.float32)

    model = TabTransformer(cat_cardinalities, num_numeric=X_num.shape[1]).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    preds = []
    with torch.no_grad():
        for i in range(0, len(val_idx), 4096):
            batch_idx = val_idx[i : i + 4096]
            xb = torch.from_numpy(X_cat[batch_idx]).long().to(device)
            xn = torch.from_numpy(scaler.transform(X_num[batch_idx]).astype(np.float32)).to(device)
            preds.append(torch.sigmoid(model(xb, xn)).cpu().numpy())
    oof_tt[val_idx] = np.concatenate(preds)
    print(f"  TabTransformer AUC: {roc_auc_score(y[val_idx], oof_tt[val_idx]):.5f}")

## 5. 블렌딩 효과 확인

In [ ]:
score_lgbm = roc_auc_score(y, oof_lgbm)
score_tt = roc_auc_score(y, oof_tt)
corr = np.corrcoef(oof_lgbm, oof_tt)[0, 1]

print(f"LightGBM 5-Fold OOF: {score_lgbm:.5f}")
print(f"TabTransformer 5-Fold OOF: {score_tt:.5f}")
print(f"OOF 예측 상관계수: {corr:.4f}")
print()
print("=== 가중평균 그리드서치 ===")
best_w, best_score = 1.0, score_lgbm
for w in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]:
    blend = w * oof_lgbm + (1 - w) * oof_tt
    blend_score = roc_auc_score(y, blend)
    marker = " <-- 현재 최고" if blend_score > best_score else ""
    if blend_score > best_score:
        best_w, best_score = w, blend_score
    print(f"LightGBM {w:.2f} + TabTransformer {1-w:.2f}: {blend_score:.5f}{marker}")

print(f"\n최적 가중치: LightGBM {best_w:.2f} + TabTransformer {1-best_w:.2f} (점수: {best_score:.5f})")
print(f"LightGBM 단독 대비 개선: {best_score - score_lgbm:+.5f}")

## 6. 결론

개선이 노이즈 수준(대략 std 0.002 미만)이라면 블렌딩은 채택할 가치가 낮습니다.
의미 있는 개선(+0.001 이상, 가능하면 다른 random_state로도 재확인)이라면,
이 가중치를 사용해서 test 예측도 같은 방식으로 블렌딩하여 제출 파일을
만들 수 있습니다 (해당 코드는 결과가 긍정적일 때 추가로 작성).
